# Place about $1 of YES wagers

This notebook uses the public `kalshi` interface to:

1. Fetch active MLB markets
2. Select one market
3. Build a dry-run YES order with about $1 of stake
4. Optionally place the real order
5. List recent orders

In [ ]:
from pathlib import Path
import sys

import pandas as pd

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "kalshi" / "__init__.py").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from kalshi import KalshiMarkets, KalshiTrading

pd.set_option("display.max_columns", 80)
pd.set_option("display.max_colwidth", 180)

## Initialize interfaces

Make sure your Jupyter kernel has the repo-level `.env` loaded.

In [ ]:
markets = KalshiMarkets()
trading = KalshiTrading.from_env()

trading.get_balance()

## Get active MLB markets

In [ ]:
active_markets = markets.list_active_mlb_markets(max_pages=10)
active_df = pd.DataFrame(markets.market_summary_rows(active_markets))

active_df[[
    "ticker",
    "title",
    "yes_bid_dollars",
    "yes_ask_dollars",
    "no_bid_dollars",
    "no_ask_dollars",
    "close_time",
]].head(25)

## Select a market

By default this chooses the first active market returned. Replace `selected_ticker` with a specific ticker from the table if you want a different one.

In [ ]:
selected_ticker = active_df.iloc[0]["ticker"]
selected_market = active_df[active_df["ticker"].eq(selected_ticker)].iloc[0]

selected_market[["ticker", "title", "yes_bid_dollars", "yes_ask_dollars", "close_time"]]

## Dry-run about $1 of YES wagers

Kalshi prices are per-contract cents. A YES contract bought at `yes_price=20` costs $0.20 and pays $1 if it resolves YES. To stake about $1, this notebook uses the selected market's current YES ask as the limit price, then computes `count = floor(100 / yes_price)`.

In [ ]:
stake_cents = 100
yes_price = int(round(float(selected_market["yes_ask_dollars"]) * 100))
yes_price = min(99, max(1, yes_price))
count = max(1, stake_cents // yes_price)
estimated_cost_dollars = count * yes_price / 100

order_kwargs = dict(
    ticker=selected_ticker,
    action="buy",
    side="yes",
    count=count,
    yes_price=yes_price,
    strategy="initial_test",
)

{
    "stake_cents": stake_cents,
    "yes_price": yes_price,
    "count": count,
    "estimated_cost_dollars": estimated_cost_dollars,
}

In [ ]:
dry_run = trading.place_order(**order_kwargs, dry_run=True)
dry_run

## Place real order

This submits a real order only if both safeguards are enabled:

- `KALSHI_ALLOW_LIVE_TRADING=true` in `.env`
- `CONFIRM_LIVE_TRADE = True` below

In [ ]:
CONFIRM_LIVE_TRADE = False

if not CONFIRM_LIVE_TRADE:
    raise RuntimeError("Set CONFIRM_LIVE_TRADE = True only when you are ready to place this real order.")

live_order = trading.place_order(**order_kwargs, dry_run=False)
live_order

## List recent orders

In [ ]:
orders = trading.get_order_history(limit=20)
orders

## Optional: tabular order view

In [ ]:
pd.json_normalize(orders.get("orders", []))